# Arabic RAG Research Prototype - `app.py` as a notebook

This notebook contains the **same code as `app.py`**, split into small parts with an explanation above each part.

**How it works**
- Each code cell starts with `%%writefile`. Running the cell writes its code into `app.py` (it does not run the app).
- The first code cell creates `app.py`; every following cell adds (`-a` = append) its part to the end of the file.
- Streamlit can only run a `.py` file, so the last cell starts the app from `app.py`.

**Always run the whole notebook from the top:** menu **Kernel → Restart & Run All** (or **Run → Run All Cells**).
Running a single cell twice would add its code to `app.py` twice.

This notebook must stay in the same folder as `app.py`, `retrieval.py` and `generation.py`.

## Part 1 - Imports and settings

Loads the libraries and our own `retrieval.py` file (which contains the original notebook retrieval functions).
- `METHODS` is the list of the four retrieval methods.
- `evaluation_questions` are the 50 test questions, read from `data/evaluation_queries_final.json`.
- `SCORE_NAMES` says what the score means for each method.

In [ ]:
%%writefile app.py
"""
app.py - Arabic RAG Research Prototype (Streamlit).

Section 1: ask a question and choose the retrieval methods.
Section 2: compare the retrieved chunks and the generated answers.

Run with:   streamlit run app.py
"""
import html
import json
import os
import time

import streamlit as st

import retrieval

METHODS = ["BM25", "BGE-M3", "Hybrid", "Hybrid + Reranking"]

# The 50 evaluation questions used in the experiments (copy of evaluation_queries_final.json)
QUESTIONS_FILE = os.path.join(os.path.dirname(__file__), "data", "evaluation_queries_final.json")
with open(QUESTIONS_FILE, "r", encoding="utf-8") as f:
    evaluation_questions = json.load(f)

OWN_QUESTION = "— Write my own question —"

# What the "score" number means for each method
SCORE_NAMES = {
    "BM25": "BM25 score",
    "BGE-M3": "cosine similarity",
    "Hybrid": "RRF score",
    "Hybrid + Reranking": "reranker score",
}

## Part 2 - Page settings and right-to-left style

Sets the browser tab title and a wide layout. The small style rule makes Arabic text boxes and answers display right-to-left.

In [ ]:
%%writefile -a app.py

st.set_page_config(page_title="Arabic RAG Research Prototype", layout="wide")

# Small style rule: Arabic text boxes and answers are written right-to-left
st.markdown(
    """
    <style>
    textarea, input[type="text"] { direction: rtl; text-align: right; }
    .arabic { direction: rtl; text-align: right; line-height: 1.9; }
    </style>
    """,
    unsafe_allow_html=True,
)

## Part 3 - Three small helper functions

- `load_models_once()` loads the chunks and the models **only once** (Streamlit remembers them with `@st.cache_resource`).
- `copy_chosen_question()` writes the question chosen from the list of 50 into the question box.
- `show_arabic(text)` displays Arabic text right-to-left.

In [ ]:
%%writefile -a app.py


@st.cache_resource(show_spinner=False)
def load_models_once():
    """Load the chunks and models only once (not on every click)."""
    retrieval.load_models()
    return True


def copy_chosen_question():
    """When a question is chosen from the list, write it into the question box."""
    choice = st.session_state["question_choice"]
    if choice != OWN_QUESTION:
        question_id = choice.split(" · ")[0]
        for item in evaluation_questions:
            if item["query_id"] == question_id:
                st.session_state["question_text"] = item["query"]


def show_arabic(text):
    """Show Arabic text right-to-left."""
    safe_text = html.escape(text).replace("\n", "<br>")
    st.markdown(f'<div class="arabic">{safe_text}</div>', unsafe_allow_html=True)

## Part 4 - Title of the page

In [ ]:
%%writefile -a app.py


# =====================================================================
# Title
# =====================================================================
st.title("Arabic RAG Research Prototype")
st.caption("Master's thesis prototype: compare four retrieval methods")

## Part 5 - Section: Ask a question

- A list of the 50 evaluation questions (optional). The chosen question is copied into the question box and can still be edited.
- The question box.
- The four methods in two rows, plus *Run all four methods*.
- *Generate answers with Gemini* and the **Gemini API key** box. A key typed in the box is used first; if the box is empty, the key saved in the `GEMINI_API_KEY` environment variable is used. The key is never saved to a file.
- The *Submit question* button.

In [ ]:
%%writefile -a app.py

# =====================================================================
# Section 1 - Ask a question
# =====================================================================
st.header("Ask a question")

question_options = [OWN_QUESTION]
for item in evaluation_questions:
    question_options.append(item["query_id"] + " · " + item["query"])

st.selectbox(
    "Choose one of the 50 evaluation questions (optional)",
    question_options,
    key="question_choice",
    on_change=copy_chosen_question,
)

question = st.text_area(
    "Question (Arabic)",
    placeholder="ما شروط القبول في برامج الدكتوراه؟",
    height=100,
    key="question_text",
)
st.caption("The chosen question appears here and can still be edited.")

st.write("Retrieval methods:")
run_all = st.checkbox("Run all four methods", value=True)

selected_methods = []
for i, method_name in enumerate(METHODS):
    if i % 2 == 0:
        method_columns = st.columns(2)  # a new row for every two methods
    with method_columns[i % 2]:
        if run_all:
            st.checkbox(method_name, value=True, disabled=True, key="all_" + method_name)
            selected_methods.append(method_name)
        else:
            if st.checkbox(method_name, key="box_" + method_name):
                selected_methods.append(method_name)

generate_answers = st.checkbox("Generate answers with Gemini", value=True)

# The API key: a key typed here is used first;
# if the box is empty, the key saved in GEMINI_API_KEY is used.
saved_key = os.environ.get("GEMINI_API_KEY", "")
typed_key = st.text_input("Gemini API key", type="password")
if saved_key:
    st.caption("Leave empty to use the key saved in GEMINI_API_KEY. A key typed here replaces it.")
else:
    st.caption("Needed only for generating answers. The key is not saved.")

if typed_key.strip() != "":
    api_key = typed_key.strip()
else:
    api_key = saved_key

submit = st.button("Submit question", type="primary")

## Part 6 - What happens when *Submit question* is pressed

1. Check that a question and at least one method were chosen.
2. Load the models (first time only).
3. Run `retrieval.run_method()` for each selected method.
4. If answers are requested and a key exists, call the original `generation.generate_rag_answer()` for each method (same model, same prompt), with the notebook's 1-second pause between calls.
5. Save the results in `st.session_state` so they stay on screen.

In [ ]:
%%writefile -a app.py

# =====================================================================
# Run retrieval (and generation) when the button is pressed
# =====================================================================
if submit:
    if question.strip() == "":
        st.warning("Please write a question first.")
    elif len(selected_methods) == 0:
        st.warning("Please choose at least one retrieval method.")
    else:
        with st.spinner("Loading the models (only the first time, this can take a few minutes)..."):
            load_models_once()

        # 1) Retrieval for each selected method
        results_by_method = {}
        for method_name in selected_methods:
            with st.spinner(f"Retrieving with {method_name}..."):
                results_by_method[method_name] = retrieval.run_method(method_name, question.strip())

        # 2) Answer generation for each method (same model and prompt for all)
        answers_by_method = {}
        if generate_answers and api_key:
            import generation

            generation.create_client(api_key)
            for method_name in selected_methods:
                with st.spinner(f"Generating the answer for {method_name}..."):
                    try:
                        answer, used_ids = generation.generate_rag_answer(
                            question.strip(), results_by_method[method_name]
                        )
                    except Exception as error:
                        answer = "ERROR: " + str(error)
                    answers_by_method[method_name] = answer
                    time.sleep(generation.SLEEP_BETWEEN_CALLS)  # same 1-second pause as the notebook
        elif generate_answers:
            st.info("Add a Gemini API key to generate answers. Retrieval results are shown below.")

        # Keep the results so they stay on screen when the page refreshes
        st.session_state["question"] = question.strip()
        st.session_state["results"] = results_by_method
        st.session_state["answers"] = answers_by_method

## Part 7 - Section: Compare results

One boxed column per method. Each column shows the generated answer (or a red error message, for example when the API key is wrong), then the 10 retrieved chunks. Each chunk row shows rank, chunk ID, document ID and score, and expands to show the full Arabic text.

For *Hybrid + Reranking* the rank and score come from the reranker (`rerank_rank`, `reranker_score`).

In [ ]:
%%writefile -a app.py

# =====================================================================
# Section 2 - Compare results
# =====================================================================
if "results" in st.session_state:
    st.header("Compare results")
    st.write("Question:")
    show_arabic(st.session_state["question"])

    results_by_method = st.session_state["results"]
    answers_by_method = st.session_state["answers"]

    result_columns = st.columns(len(results_by_method))
    for column, method_name in zip(result_columns, results_by_method):
        with column:
            with st.container(border=True):
                st.subheader(method_name)

                # Generated answer
                st.markdown("**Generated answer**")
                if method_name in answers_by_method:
                    answer = answers_by_method[method_name]
                    if answer.startswith("ERROR:"):
                        st.error(answer)  # for example, a wrong or expired API key
                    else:
                        show_arabic(answer)
                else:
                    st.caption("No answer generated.")

                # Retrieved chunks (top-10), each one in an expandable area
                st.markdown(f"**Retrieved chunks** (score = {SCORE_NAMES[method_name]})")
                for result in results_by_method[method_name]:
                    # For reranking, the rank and score come from the reranker
                    if method_name == "Hybrid + Reranking":
                        rank = result["rerank_rank"]
                        score = result["reranker_score"]
                    else:
                        rank = result["rank"]
                        score = result["score"]

                    label = f"#{rank}  {result['chunk_id']}  |  {result['document_id']}  |  {score:.4f}"
                    with st.expander(label):
                        st.caption("Section: " + result["section"])
                        show_arabic(result["text"])

    st.caption("All ten chunks shown for each method are the context given to Gemini (TOP_K_CHUNKS = 10).")

## Check that `app.py` is complete

This cell checks that the file written above is valid Python.

In [ ]:
import py_compile

py_compile.compile("app.py", doraise=True)
print("app.py is ready")

## Start the app

The browser opens a new tab at http://localhost:8501.

The cell keeps showing `[*]` while the app runs - this is normal. To stop the app, press the **■ (Interrupt)** button at the top of Jupyter.

In [ ]:
!streamlit run app.py